# Integration NumPy, Pandas et Matplotlib



## 0. Imports et Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style global dark
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1f2e',
    'axes.edgecolor':   '#2e3448',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2e3448',
    'grid.alpha':       0.6,
    'legend.facecolor': '#1a1f2e',
    'legend.edgecolor': '#2e3448',
})
PALETTE = ['#4FC3F7','#81C784','#FFD54F','#FF8A65','#CE93D8',
           '#F48FB1','#80DEEA','#FFAB40','#B0BEC5','#EF9A9A']

print(f"numpy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"matplotlib {plt.matplotlib.__version__}")
print(f"seaborn    {sns.__version__}")


## Tache 1 — Preparation des Donnees

**Objectif** : generer un dataset synthetique realiste de temperatures mensuelles
pour 10 villes mondiales avec `np.random.uniform`, puis le convertir en DataFrame Pandas.




In [ ]:
np.random.seed(42)

# Noms des villes et leur profil climatique (mu mensuelle, amplitude saisonniere)
cities = [
    'Paris',        # temperee oceanique
    'Moscou',       # continentale froide
    'Dubai',        # desertique chaude
    'Sydney',       # temperee hemispere sud
    'Tokyo',        # temperee humide
    'Lagos',        # tropicale
    'Montreal',     # continentale froide hemispere nord
    'Singapore',    # equatoriale
    'Cairo',        # aride chaude
    'Reykjavik',    # subarctique
]

months = ['Jan','Feb','Mar','Apr','May','Jun',
          'Jul','Aug','Sep','Oct','Nov','Dec']

# Profil climatique : (temp_base_annuelle, amplitude_saisonniere)
#  amplitude > 0 -> ete chaud en juillet, hiver froid en janvier
#  amplitude < 0 -> inverse (hemispere sud)
profiles = {
    'Paris':      ( 11,  16),
    'Moscou':     (  4,  30),
    'Dubai':      ( 29,  12),
    'Sydney':     ( 18, -12),   # hemispere sud -> inverse
    'Tokyo':      ( 15,  20),
    'Lagos':      ( 27,   4),
    'Montreal':   (  7,  30),
    'Singapore':  ( 27,   2),
    'Cairo':      ( 22,  18),
    'Reykjavik':  (  5,  14),
}

# Generation : sinusoide climatique + bruit aleatoire
angle = np.linspace(0, 2*np.pi, 12, endpoint=False)

temp_data = {}
for city, (base, amplitude) in profiles.items():
    # Phase : juillet = pic (mois 6, indice 6)
    seasonal = amplitude / 2 * np.sin(angle - np.pi/2 + np.pi)  # pic en juillet
    noise    = np.random.uniform(-2.5, 2.5, 12)
    temps    = np.clip(base + seasonal + noise, -5, 35).round(1)
    temp_data[city] = temps

# Conversion en tableau NumPy
temp_array = np.array([temp_data[c] for c in cities])
print(f"Tableau NumPy - shape : {temp_array.shape}  (villes x mois)")
print(f"Plage de temperatures : {temp_array.min():.1f}°C  ->  {temp_array.max():.1f}°C")
print()

# Conversion en DataFrame Pandas
df = pd.DataFrame(
    data    = temp_array,
    index   = cities,
    columns = months
)
df.index.name   = 'City'
df.columns.name = 'Month'

print("DataFrame Pandas :")
display(df.round(1))


## Tache 2 — Analyse des Donnees

In [ ]:
# --- Temperature moyenne annuelle par ville ---
df['Annual_Mean'] = df[months].mean(axis=1).round(2)

# --- Statistiques mensuelles ---
monthly_stats = pd.DataFrame({
    'Mean':  df[months].mean(axis=0).round(2),
    'Std':   df[months].std(axis=0).round(2),
    'Min':   df[months].min(axis=0).round(2),
    'Max':   df[months].max(axis=0).round(2),
}, index=months)

# --- Villes extremes ---
hottest_city  = df['Annual_Mean'].idxmax()
coldest_city  = df['Annual_Mean'].idxmin()
hottest_temp  = df['Annual_Mean'].max()
coldest_temp  = df['Annual_Mean'].min()

# --- Amplitude thermique annuelle par ville ---
df['Amplitude'] = (df[months].max(axis=1) - df[months].min(axis=1)).round(2)

# --- Mois le plus chaud / froid globalement ---
hottest_month = df[months].mean(axis=0).idxmax()
coldest_month = df[months].mean(axis=0).idxmin()

print("=" * 55)
print("RESUME ANALYTIQUE")
print("=" * 55)
print(f"  Ville la plus chaude  : {hottest_city:<12} ({hottest_temp:.2f}°C annuel)")
print(f"  Ville la plus froide  : {coldest_city:<12} ({coldest_temp:.2f}°C annuel)")
print(f"  Mois le plus chaud    : {hottest_month} (moy. mondiale : {df[months].mean(axis=0)[hottest_month]:.1f}°C)")
print(f"  Mois le plus froid    : {coldest_month} (moy. mondiale : {df[months].mean(axis=0)[coldest_month]:.1f}°C)")
print()
print("Temperatures moyennes annuelles (triees) :")
print(df['Annual_Mean'].sort_values(ascending=False).to_frame().to_string())
print()
print("Amplitude thermique annuelle par ville :")
print(df['Amplitude'].sort_values(ascending=False).to_frame().to_string())


In [ ]:
# --- Statistiques descriptives completes ---
print("Statistiques descriptives par mois (toutes villes confondues) :")
display(monthly_stats)

print("\nCorrelation entre les villes (similitude des profils climatiques) :")
corr_matrix = df[months].T.corr().round(3)
display(corr_matrix)


## Tache 3 — Visualisations

### 3.1 Tableau de bord principal

In [ ]:
fig = plt.figure(figsize=(22, 16))
fig.patch.set_facecolor('#0f1117')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

# ── 1. Line chart : toutes les villes sur 12 mois ─────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
for i, city in enumerate(cities):
    lw    = 3.0 if city in [hottest_city, coldest_city] else 1.6
    alpha = 1.0 if city in [hottest_city, coldest_city] else 0.75
    ax1.plot(months, df.loc[city, months],
             color=PALETTE[i], lw=lw, alpha=alpha,
             marker='o', markersize=4 if lw==1.6 else 7,
             label=city)
ax1.set_title('Temperatures mensuelles par ville (12 mois)', fontweight='bold', fontsize=12)
ax1.set_xlabel('Mois'); ax1.set_ylabel('Temperature (°C)')
ax1.axhline(0, color='white', lw=1, linestyle='--', alpha=0.3)
ax1.legend(fontsize=8, ncol=2, loc='upper right')

# ── 2. Bar chart : temperature annuelle moyenne ────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
annual_sorted = df['Annual_Mean'].sort_values(ascending=True)
colors_bar    = ['#FF8A65' if c == hottest_city
                 else '#4FC3F7' if c == coldest_city
                 else '#81C784' for c in annual_sorted.index]
bars = ax2.barh(annual_sorted.index, annual_sorted.values,
                color=colors_bar, edgecolor='#0f1117', alpha=0.88)
for bar, val in zip(bars, annual_sorted.values):
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}°C', va='center', fontsize=8.5,
             color='#FFD54F', fontweight='bold')
ax2.set_title('Moyenne annuelle par ville', fontweight='bold', fontsize=12)
ax2.set_xlabel('Temperature (°C)')
ax2.axvline(0, color='white', lw=1, linestyle='--', alpha=0.3)

# ── 3. Heatmap : matrice temperature ──────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
im = ax3.imshow(df[months].values, aspect='auto', cmap='RdYlBu_r',
                vmin=-5, vmax=35, interpolation='nearest')
plt.colorbar(im, ax=ax3, label='Temperature (°C)', shrink=0.9)
ax3.set_xticks(range(12));  ax3.set_xticklabels(months, fontsize=9)
ax3.set_yticks(range(len(cities))); ax3.set_yticklabels(cities, fontsize=9)
for i in range(len(cities)):
    for j in range(12):
        val   = df[months].values[i, j]
        color = 'black' if 5 < val < 28 else 'white'
        ax3.text(j, i, f'{val:.0f}', ha='center', va='center',
                 fontsize=7.5, color=color, fontweight='bold')
ax3.set_title('Heatmap des Temperatures (°C)', fontweight='bold', fontsize=12)

# ── 4. Box plot : distribution par ville ──────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
bp  = ax4.boxplot(
    [df.loc[c, months].values for c in cities],
    patch_artist=True,
    vert=False,
    medianprops=dict(color='white', lw=2),
    whiskerprops=dict(color='#e0e0e0'),
    capprops=dict(color='#e0e0e0'),
    flierprops=dict(marker='o', color='#e0e0e0', alpha=0.5, markersize=4),
)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.72)
ax4.set_yticks(range(1, len(cities)+1))
ax4.set_yticklabels(cities, fontsize=8)
ax4.axvline(0, color='white', lw=1, linestyle='--', alpha=0.3)
ax4.set_title('Distribution des temperatures', fontweight='bold', fontsize=12)
ax4.set_xlabel('Temperature (°C)')

# ── 5. Amplitude thermique ────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
amp_sorted = df['Amplitude'].sort_values(ascending=False)
c_amp = [PALETTE[cities.index(c)] for c in amp_sorted.index]
bars5 = ax5.bar(amp_sorted.index, amp_sorted.values,
                color=c_amp, edgecolor='#0f1117', alpha=0.88)
for b in bars5:
    ax5.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
             f'{b.get_height():.1f}°', ha='center', fontsize=8, color='#FFD54F')
ax5.set_title('Amplitude thermique annuelle (°C)', fontweight='bold', fontsize=12)
ax5.set_xlabel('Ville'); ax5.set_ylabel('Amplitude (°C)')
ax5.tick_params(axis='x', rotation=45)

# ── 6. Moy. mondiale mensuelle + ecart-type ───────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
world_mean = df[months].mean(axis=0)
world_std  = df[months].std(axis=0)
ax6.plot(months, world_mean, 'o-', color='#4FC3F7', lw=2.5, ms=7, label='Moy. mondiale')
ax6.fill_between(range(12),
                 world_mean - world_std,
                 world_mean + world_std,
                 alpha=0.22, color='#4FC3F7', label='±1 ecart-type')
ax6.set_title('Moyenne mondiale mensuelle ± std', fontweight='bold', fontsize=12)
ax6.set_xlabel('Mois'); ax6.set_ylabel('Temperature (°C)')
ax6.set_xticks(range(12)); ax6.set_xticklabels(months, fontsize=8)
ax6.legend(fontsize=9)

# ── 7. Heatmap correlation entre villes ──────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
corr = df[months].T.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, ax=ax7, cmap='coolwarm', vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size':6},
            linewidths=0.3, linecolor='#0f1117',
            cbar_kws={'shrink':0.8})
ax7.set_title('Correlation inter-villes', fontweight='bold', fontsize=12)
ax7.tick_params(axis='x', rotation=45, labelsize=7)
ax7.tick_params(axis='y', rotation=0,  labelsize=7)

fig.suptitle('Analyse des Tendances de Temperature Mondiale — 10 Villes / 12 Mois',
             color='#e0e0e0', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('temperature_dashboard.png', dpi=145, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Tableau de bord principal genere")


### 3.2 Analyse approfondie : tendances et comparaisons

In [ ]:
fig2, axes = plt.subplots(2, 2, figsize=(16, 12))
fig2.patch.set_facecolor('#0f1117')

# ── 1. Villes extremes : profils annuels ─────────────────────────────────
ax = axes[0, 0]
highlight = [hottest_city, coldest_city, 'Paris', 'Moscou']
hl_styles = [
    ('#FF8A65', '-',  2.8, 'o', f'{hottest_city} (PLUS CHAUDE)'),
    ('#4FC3F7', '-',  2.8, 's', f'{coldest_city} (PLUS FROIDE)'),
    ('#FFD54F', '--', 2.0, '^', 'Paris'),
    ('#CE93D8', '--', 2.0, 'D', 'Moscou'),
]
for city, (color, ls, lw, mk, label) in zip(highlight, hl_styles):
    ax.plot(months, df.loc[city, months], color=color, lw=lw,
            ls=ls, marker=mk, ms=7, label=label)
ax.fill_between(range(12), df[months].max(axis=0), df[months].min(axis=0),
                alpha=0.08, color='white', label='Plage totale')
ax.axhline(0, color='white', lw=1, ls=':', alpha=0.4)
ax.set_title('Villes extremes vs references', fontweight='bold')
ax.set_xlabel('Mois'); ax.set_ylabel('Temperature (°C)')
ax.set_xticks(range(12)); ax.set_xticklabels(months, fontsize=9)
ax.legend(fontsize=9)

# ── 2. Stacked area : repartition par saison ─────────────────────────────
ax = axes[0, 1]
seasons = {'Hiver': ['Dec','Jan','Feb'], 'Printemps': ['Mar','Apr','May'],
           'Ete':   ['Jun','Jul','Aug'], 'Automne':   ['Sep','Oct','Nov']}
sea_means = {s: df[mois].mean(axis=1).values for s, mois in seasons.items()}
sea_colors = ['#4FC3F7','#81C784','#FF8A65','#FFD54F']
x = np.arange(len(cities))
bottoms = np.zeros(len(cities))
for (season, vals), color in zip(sea_means.items(), sea_colors):
    ax.bar(x, vals, bottom=bottoms, color=color, alpha=0.82,
           edgecolor='#0f1117', lw=0.5, label=season, width=0.7)
    bottoms += vals
ax.set_xticks(x); ax.set_xticklabels(cities, rotation=40, ha='right', fontsize=8)
ax.set_title('Contribution saisonniere par ville', fontweight='bold')
ax.set_ylabel('Temperature cumulee (°C)')
ax.legend(fontsize=9, loc='upper right')

# ── 3. Scatter : moyenne annuelle vs amplitude ─────────────────────────────
ax = axes[1, 0]
scatter_colors = PALETTE[:len(cities)]
sc = ax.scatter(df['Annual_Mean'], df['Amplitude'],
                c=scatter_colors, s=200, alpha=0.88,
                edgecolors='white', linewidths=1.5, zorder=5)
for i, city in enumerate(cities):
    ax.annotate(city,
                (df.loc[city,'Annual_Mean'], df.loc[city,'Amplitude']),
                textcoords='offset points', xytext=(8, 4),
                fontsize=8.5, color=PALETTE[i])
ax.set_title('Moyenne annuelle vs Amplitude thermique', fontweight='bold')
ax.set_xlabel('Temperature annuelle moyenne (°C)')
ax.set_ylabel('Amplitude thermique (°C)')
ax.axvline(df['Annual_Mean'].mean(), color='white', lw=1.2,
           ls='--', alpha=0.5, label=f"Moy. = {df['Annual_Mean'].mean():.1f}°C")
ax.axhline(df['Amplitude'].mean(), color='#FFD54F', lw=1.2,
           ls='--', alpha=0.5, label=f"Ampl. moy. = {df['Amplitude'].mean():.1f}°C")
ax.legend(fontsize=9)

# ── 4. Violinplot distribution par hemisphere ─────────────────────────────
ax = axes[1, 1]
north_cities = ['Paris','Moscou','Dubai','Tokyo','Lagos','Montreal','Singapore','Cairo','Reykjavik']
south_cities = ['Sydney']
all_north = df.loc[north_cities, months].values.flatten()
all_south = df.loc[south_cities, months].values.flatten()
parts = ax.violinplot([all_north, all_south], positions=[1,2],
                      showmedians=True, showmeans=True)
for pc, color in zip(parts['bodies'], ['#4FC3F7','#FFD54F']):
    pc.set_facecolor(color); pc.set_alpha(0.65)
parts['cmedians'].set_color('white'); parts['cmedians'].set_lw(2.5)
parts['cmeans'].set_color('#FF8A65'); parts['cmeans'].set_lw(2)
ax.set_xticks([1,2])
ax.set_xticklabels(['Hemisphere Nord
(9 villes)', 'Hemisphere Sud
(Sydney)'], fontsize=10)
ax.set_title('Distribution temperatures Nord vs Sud', fontweight='bold')
ax.set_ylabel('Temperature (°C)')
ax.axhline(0, color='white', lw=1, ls='--', alpha=0.3)

fig2.suptitle('Analyse Approfondie — Tendances Climatiques Mondiales',
              color='#e0e0e0', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('temperature_advanced.png', dpi=145, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Visualisations approfondies generees")


## Tache 4 — Rapport de Synthese

In [ ]:
from scipy.stats import pearsonr

print("=" * 60)
print("RAPPORT DE SYNTHESE — ANALYSE TEMPERATURES MONDIALES")
print("=" * 60)

print("\n1. STATISTIQUES GLOBALES")
print("-" * 40)
print(f"   Villes analysees     : {len(cities)}")
print(f"   Periode couverte     : 12 mois")
print(f"   Temperature minimale : {df[months].values.min():.1f}°C")
print(f"   Temperature maximale : {df[months].values.max():.1f}°C")
print(f"   Moyenne mondiale     : {df[months].values.mean():.2f}°C")

print("\n2. VILLES EXTREMES")
print("-" * 40)
print(f"   Plus chaude : {hottest_city} ({hottest_temp:.2f}°C)")
print(f"   Plus froide : {coldest_city} ({coldest_temp:.2f}°C)")
print(f"   Ecart       : {hottest_temp - coldest_temp:.2f}°C")

print("\n3. TENDANCES SAISONNIERES")
print("-" * 40)
world_mean = df[months].mean(axis=0)
print(f"   Mois le plus chaud  : {world_mean.idxmax()} ({world_mean.max():.1f}°C)")
print(f"   Mois le plus froid  : {world_mean.idxmin()} ({world_mean.min():.1f}°C)")
print(f"   Amplitude mondiale  : {world_mean.max() - world_mean.min():.1f}°C")

print("\n4. PROFILS CLIMATIQUES")
print("-" * 40)
for city in df.sort_values('Amplitude', ascending=False).index:
    amp  = df.loc[city,'Amplitude']
    mean = df.loc[city,'Annual_Mean']
    typ  = ('Continental' if amp > 20
            else 'Tropical/Equatorial' if mean > 24
            else 'Aride/Desert' if mean > 18 and amp < 15
            else 'Temperee' if 8 < mean < 18
            else 'Subarctique/Polaire')
    print(f"   {city:<12} : {mean:>5.1f}°C moy. | {amp:>5.1f}°C amplitude | {typ}")

print("\n5. CORRELATIONS NOTABLES")
print("-" * 40)
r_ma, p_ma = pearsonr(df['Annual_Mean'], df['Amplitude'])
print(f"   Moy. annuelle ~ Amplitude : r={r_ma:.4f}  p={p_ma:.4f}")
print(f"   -> {'Correlation negative' if r_ma<0 else 'Correlation positive'}")
print("      (les villes froides ont une plus grande amplitude thermique)")

corr_mat = df[months].T.corr()
pairs = []
for i in range(len(cities)):
    for j in range(i+1, len(cities)):
        pairs.append((cities[i], cities[j], corr_mat.iloc[i,j]))
pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print("\n   Paires de villes les plus correlees :")
for c1, c2, r in pairs[:3]:
    print(f"     {c1} <-> {c2} : r={r:.3f}")
print("\n   Paires de villes les moins correlees :")
for c1, c2, r in sorted(pairs, key=lambda x: x[2])[:3]:
    print(f"     {c1} <-> {c2} : r={r:.3f}")

print("\n6. OBSERVATIONS ET INSIGHTS")
print("-" * 40)
print("   a) Les villes tropicales (Lagos, Singapore, Dubai) ont une")
print("      amplitude thermique faible (<10°C) malgre des moyennes elevees.")
print("   b) Les villes continentales (Moscou, Montreal) presentent les")
print("      plus grandes amplitudes (>25°C), typiques du climat continental.")
print("   c) Sydney montre un profil inverse (hiver en juillet) par rapport")
print("      aux villes de l'hemisphere nord, illustrant l'inversement saisonnier.")
print("   d) Reykjavik, la ville la plus froide, reste habitable grace a")
print("      l'effet moderateur de l'ocean Atlantique (amplitude moderee).")
